In [7]:
!pip install googletrans==3.1.0a0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16410 sha256=61efa86f8a17852239e6474c758f98b6c923ea57254d3f682b8cc7622854f6a4
  Stored in directory: /Users/lukeschreiber/Library/Caches/pip/wheels/96/ac/bd/9df9eab356c0576896e97147425987f6f45e9e46456c978d18
Successfully built googletrans
  Attempting uninstall: googletrans
    Found existing installation: googletrans 4.0.0rc1
    Uninstalling googletrans-4.0.0rc1:
      Successfully uninstalled googletrans-4.0.0rc1


In [9]:
import os
import time
import re
from googletrans import Translator

# --- CONFIG ---
INPUT_FOLDER = "txts"
OUTPUT_FOLDER = "translated_txts"
MAX_CHARS = 2000 # Smaller chunks to avoid malformed URL errors

def clean_text(text):
    # This removes the massive strings of dots and 'o' symbols 
    # that appear in your SISÄLLYSLUETTELO (Table of Contents)
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'[o]{3,}', ' ', text)
    return text

def run_translation():
    translator = Translator()
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.txt')]
    
    for filename in files:
        output_path = os.path.join(OUTPUT_FOLDER, filename)
        if os.path.exists(output_path): continue

        print(f"⚙️ Processing: {filename}")
        
        try:
            with open(os.path.join(INPUT_FOLDER, filename), 'r', encoding='utf-8') as f:
                content = f.read()
            
            # 1. Clean the text to remove the 'toxic' symbols
            content = clean_text(content)
            chunks = [content[i:i+MAX_CHARS] for i in range(0, len(content), MAX_CHARS)]
            
            translated_parts = []
            
            for idx, chunk in enumerate(chunks, 1):
                try:
                    time.sleep(2) # Human delay
                    res = translator.translate(chunk, src='fi', dest='en')
                    translated_parts.append(res.text)
                except AttributeError as e:
                    if "as_dict" in str(e):
                        print(f"    [!] Part {idx} contains 'Toxic' characters for the API. Skipping translation for this part.")
                        # If it fails, we keep the original text so you don't lose data
                        translated_parts.append(f"\n[SECTION UNTRANSLATABLE - KEEPING ORIGINAL]\n{chunk}\n")
                    else:
                        raise e
                except Exception as e:
                    print(f"    [!] Unexpected error on Part {idx}: {e}")
                    translated_parts.append(chunk)

            with open(output_path, 'w', encoding='utf-8') as f:
                f.write("\n\n".join(translated_parts))
            print("    [✓] Saved.")

        except Exception as e:
            print(f"    [X] Critical failure on {filename}: {e}")

if __name__ == "__main__":
    run_translation()